# 02 — Pré-processamento

**Dimensão 4 da rúbrica — 15 pontos.**

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

In [302]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

RAW = Path("..") / "data" / "raw" / "df.csv"
PROCESSED = Path("..") / "data" / "processed"
TARGET = "COMPLIANCE"

pd.set_option("display.max_columns", None)

In [303]:
df = pd.read_csv(RAW)
df.head()

,ID,MONTHS_BALANCE,STATUS,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,COMPLIANCE
0,5008804,0,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2,0
1,5008804,-1,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2,0
2,5008804,-2,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2,0
3,5008804,-3,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2,0
4,5008804,-4,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2,0


## 1. Dados faltantes

In [304]:
# Identificando colunas com valores nulos
df.isnull().sum()

ID                          0
MONTHS_BALANCE              0
STATUS                      0
CODE_GENDER                 0
FLAG_OWN_CAR                0
FLAG_OWN_REALTY             0
CNT_CHILDREN                0
AMT_INCOME_TOTAL            0
NAME_INCOME_TYPE            0
NAME_EDUCATION_TYPE         0
NAME_FAMILY_STATUS          0
NAME_HOUSING_TYPE           0
DAYS_BIRTH                  0
DAYS_EMPLOYED               0
FLAG_MOBIL                  0
FLAG_WORK_PHONE             0
FLAG_PHONE                  0
FLAG_EMAIL                  0
OCCUPATION_TYPE        240048
CNT_FAM_MEMBERS             0
COMPLIANCE                  0
dtype: int64

----

***Tratando os nulos da coluna OCCUPATION_TYPE***

**Decisão:** 

Baseado no dicionário de dados, se o número na coluna DAYS_EMPLOYED for positivo, significa que a pessoa está desempregada.

Para essa condição, os nulos da coluna OCCUPATION_TYPE foram substituídos por "Unemployed".

Já para o restante dos nulos, foram substituir por "Unknown", pois não se tem informações sobre a ocupação dessas pessoas.

In [305]:
for item in df[df.DAYS_EMPLOYED > 0].index:
  df.loc[item, "OCCUPATION_TYPE"] = "Unemployed"


df.fillna('Unknown', inplace=True)

df.OCCUPATION_TYPE.isnull().sum()

np.int64(0)

### 1.1 Excluindo colunas que não serão mais utilizadas

*Excuindo as colunas MONTHS_BALANCE e STATUS, que não serão mais utilizadas e a coluna FLAG_MOBIL que contém apenas 1 valor*

In [306]:
df.drop(columns=['MONTHS_BALANCE', 'STATUS', 'FLAG_MOBIL'], inplace=True)
df.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,COMPLIANCE
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
1,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
2,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
3,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
4,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0


*Removendo os valores duplicados*

In [307]:
df.drop_duplicates(inplace=True)
df.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,COMPLIANCE
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
16,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
31,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,0,0,0,Security staff,2,0
61,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,0,1,1,Sales staff,1,0
66,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,0,1,1,Sales staff,1,0


*Excluindo coluna ID*

In [ ]:
#df = df.set_index('ID')
df.drop(columns='ID', inplace=True, )
df.reset_index(inplace=True)
df.head()

,index,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,COMPLIANCE
0,0,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
1,16,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
2,31,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,0,0,0,Security staff,2,0
3,61,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,0,1,1,Sales staff,1,0
4,66,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,0,1,1,Sales staff,1,0


In [309]:
df.drop(columns='index', inplace=True)
df.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,COMPLIANCE
0,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
1,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
2,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,0,0,0,Security staff,2,0
3,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,0,1,1,Sales staff,1,0
4,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,0,1,1,Sales staff,1,0


In [310]:
df.shape

(36457, 17)

## 2. Definição da variável alvo

*Variável alvo e Variáveis independentes*

In [ ]:
# Variável alvo é a COMPLIANCE
y = df[TARGET]

# Independentes
x = df.drop(columns=TARGET)
x.columns

Index(['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN',
       'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
       'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH',
       'DAYS_EMPLOYED', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL',
       'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS'],
      dtype='str')

## 3. Normalização / padronização

**Escolha do Escalonador**


Para modelos baseados em árvores não há necessidade de escalonador, porém, para modelos baseados em distância é recomendado sua utilização.

Foi escolhido o **StandardScaler** para as colunas **numéricas**, pois essas colunas apresentam valores bem diferentes.

Aplicando o StandardScaler elas ficam todas com a mesma escala (com média para 0 e desvio padrão para 1).

---

*Tratando a coluna DAYS_EMPLOYED*

Antes de aplicar o escalonador, há necessidade de tratar os valores da coluna DAYS_EMPLOYED.

Pelo histograma da etapa 1.2, verifica-se que há valores positivos bem discrepantes e que o range maior é para os positivos. 

*Verificando os valores únicos positivos*

In [ ]:
df.DAYS_EMPLOYED[df.DAYS_EMPLOYED > 0].unique()

array([365243])

Nota-se que há apenas 1 valor positivo (365243) para vários clientes (indicando que o cliente está desempregado) o que pode se tratar que é uma espécie de flag.

*Alterando os valores positivos da coluna DAYS_EMPLOYED para 1, para não afetar os testes.*

In [ ]:
df.DAYS_EMPLOYED = df.DAYS_EMPLOYED.apply(lambda x: 1 if x > 0 else x)
df.DAYS_EMPLOYED[df.DAYS_EMPLOYED > 0].unique()

array([1])

----

*Aplicando o **StandardScaler** nas variáveis numéricas de interesse*

Aplicando o StandardScaler nas variáveis numéricas "AMT_INCOME_TOTAL", "DAYS_BIRTH", "DAYS_EMPLOYED", "CNT_CHILDREN" e "CNT_FAM_MEMBERS"

In [ ]:
# Separando as colunas numéricas
numericas = ['AMT_INCOME_TOTAL', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS']

# Instanciando o escalonador
scaler = StandardScaler()

# Escalonando
df_scaled = scaler.fit_transform(df[numericas])

# Transformando em dataframe
df_scaled = pd.DataFrame(df_scaled, columns=numericas)
#df_scaled.set_index(df.ID)

df_scaled.shape

(36457, 5)

*Excluindo as colunas numéricas do df original*

In [ ]:
df.drop(columns=df[numericas], inplace = True)
df.shape

(36457, 12)

*Juntando os dataframes*

In [ ]:
df = pd.concat([df, df_scaled], axis = 1)
df.shape

(71189, 17)

In [ ]:
df.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,COMPLIANCE,AMT_INCOME_TOTAL,DAYS_BIRTH,DAYS_EMPLOYED,CNT_CHILDREN,CNT_FAM_MEMBERS
0,M,Y,Y,Working,Higher education,Civil marriage,Rented apartment,1.0,0.0,0.0,Unknown,0.0,2.365845,0.945169,-0.989408,-0.579661,-0.217680
16,M,Y,Y,Working,Higher education,Civil marriage,Rented apartment,1.0,0.0,0.0,Unknown,0.0,-0.507779,-0.429194,0.425088,-0.579661,-0.217680
31,M,Y,Y,Working,Secondary / secondary special,Married,House / apartment,0.0,0.0,0.0,Security staff,0.0,2.144797,0.983974,0.077801,0.767400,0.879204
61,F,N,Y,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0.0,1.0,1.0,Sales staff,0.0,-0.596198,0.848513,0.180466,2.114462,1.976088
66,F,N,Y,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0.0,1.0,1.0,Sales staff,0.0,-0.596198,0.848513,0.180466,2.114462,1.976088


## 4. Feature engineering

**Justificativa:**

Há colunas que podem ser binarizadas e as colunas categóricas podem ser tratadas para ficarem com o mesmo padrão das outras colunas.

Para isso, foram escolhidos o **Label Encoder** e o **One-Hot Encoder**.

_____

**Verificando as colunas que podem ser binarizadas**

In [ ]:
# Verificando os valores únicos de cada coluna
for col in df.columns:
  if df[col].nunique() == 2:
    print(f"Coluna: {col}, Valores únicos: {df[col].unique()}")

Coluna: CODE_GENDER, Valores únicos: <StringArray>
['M', 'F', nan]
Length: 3, dtype: str
Coluna: FLAG_OWN_CAR, Valores únicos: <StringArray>
['Y', 'N', nan]
Length: 3, dtype: str
Coluna: FLAG_OWN_REALTY, Valores únicos: <StringArray>
['Y', 'N', nan]
Length: 3, dtype: str
Coluna: FLAG_WORK_PHONE, Valores únicos: [ 1.  0. nan]
Coluna: FLAG_PHONE, Valores únicos: [ 0.  1. nan]
Coluna: FLAG_EMAIL, Valores únicos: [ 0.  1. nan]
Coluna: COMPLIANCE, Valores únicos: [ 0.  1. nan]


As colunas CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALTY, FLAG_WORK_PHONE, FLAG_PHONE, e FLAG_EMAIL possuem apenas 2 valores, porém apenas 3 delas (CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALTY) não são binarizadas.

Para binarizá-las, será utilizado o **Label Encoder**.

In [ ]:
# Separando as colunas
colunas = ['CODE_GENDER','FLAG_OWN_CAR','FLAG_OWN_REALTY']

# Aplicando o label_encorder
label_encoder = LabelEncoder()
for i in colunas:
  df[i] = label_encoder.fit_transform(df[i])
df.head()

# Legenda: M = 1, F = 0, Y = 1, N = 0

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,COMPLIANCE,AMT_INCOME_TOTAL,DAYS_BIRTH,DAYS_EMPLOYED,CNT_CHILDREN,CNT_FAM_MEMBERS
0,1,1,1,Working,Higher education,Civil marriage,Rented apartment,1.0,0.0,0.0,Unknown,0.0,2.365845,0.945169,-0.989408,-0.579661,-0.217680
16,1,1,1,Working,Higher education,Civil marriage,Rented apartment,1.0,0.0,0.0,Unknown,0.0,-0.507779,-0.429194,0.425088,-0.579661,-0.217680
31,1,1,1,Working,Secondary / secondary special,Married,House / apartment,0.0,0.0,0.0,Security staff,0.0,2.144797,0.983974,0.077801,0.767400,0.879204
61,0,0,1,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0.0,1.0,1.0,Sales staff,0.0,-0.596198,0.848513,0.180466,2.114462,1.976088
66,0,0,1,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0.0,1.0,1.0,Sales staff,0.0,-0.596198,0.848513,0.180466,2.114462,1.976088


In [ ]:
df.shape

(71189, 17)

----

**Aplicando o One-Hot Encoder nas colunas categóricas**

In [ ]:
# Separando as colunas categóricas
categoricas = []

for i in df.columns:
  if df[i].dtype == 'str':
    categoricas.append(i)


# Aplicando o hot encoder
hot = []

for i in df.columns:
  hot = pd.get_dummies(df[categoricas], prefix = 'hot')


# Mesclando os  dataframes
df = pd.concat([df, hot], axis=1)
df.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,COMPLIANCE,AMT_INCOME_TOTAL,DAYS_BIRTH,DAYS_EMPLOYED,CNT_CHILDREN,CNT_FAM_MEMBERS,hot_Commercial associate,hot_Pensioner,hot_State servant,hot_Student,hot_Working,hot_Academic degree,hot_Higher education,hot_Incomplete higher,hot_Lower secondary,hot_Secondary / secondary special,hot_Civil marriage,hot_Married,hot_Separated,hot_Single / not married,hot_Widow,hot_Co-op apartment,hot_House / apartment,hot_Municipal apartment,hot_Office apartment,hot_Rented apartment,hot_With parents,hot_Accountants,hot_Cleaning staff,hot_Cooking staff,hot_Core staff,hot_Drivers,hot_HR staff,hot_High skill tech staff,hot_IT staff,hot_Laborers,hot_Low-skill Laborers,hot_Managers,hot_Medicine staff,hot_Private service staff,hot_Realty agents,hot_Sales staff,hot_Secretaries,hot_Security staff,hot_Unemployed,hot_Unknown,hot_Waiters/barmen staff
0,1,1,1,Working,Higher education,Civil marriage,Rented apartment,1.0,0.0,0.0,Unknown,0.0,2.365845,0.945169,-0.989408,-0.579661,-0.217680,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
16,1,1,1,Working,Higher education,Civil marriage,Rented apartment,1.0,0.0,0.0,Unknown,0.0,-0.507779,-0.429194,0.425088,-0.579661,-0.217680,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
31,1,1,1,Working,Secondary / secondary special,Married,House / apartment,0.0,0.0,0.0,Security staff,0.0,2.144797,0.983974,0.077801,0.767400,0.879204,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
61,0,0,1,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0.0,1.0,1.0,Sales staff,0.0,-0.596198,0.848513,0.180466,2.114462,1.976088,True,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False
66,0,0,1,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0.0,1.0,1.0,Sales staff,0.0,-0.596198,0.848513,0.180466,2.114462,1.976088,True,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False


In [ ]:
df.drop(columns=['index', 'ID'], inplace=True)

KeyError: "['index', 'ID'] not found in axis"

In [ ]:
df.shape

## 5. Salvar dataset tratado

In [ ]:
dataset_tratado = df.copy

PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED / "dataset_tratado.csv", index=False)